In [10]:
#!/usr/bin/env python
# coding: utf-8

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from stargazer.stargazer import Stargazer
from datetime import timedelta, datetime
from IPython.display import display, Latex, Math, HTML
from linearmodels.panel import PanelOLS
import addfips
import warnings
import os

# Suppress warnings
warnings.filterwarnings('ignore')

#Read in data files
zillow_county = pd.read_csv('Data/zillow county.csv')
employment_levels = pd.read_csv('Data/employment_data_2017_2019_combined.csv')
tariffs = pd.read_csv('Data/ustariffs-by-county.csv')
population = pd.read_csv('Data/population_data.csv')

# Begin Editing population file
population.rename(columns={'Unnamed: 0': 'County'}, inplace=True)
population = population[['County','2017','2018','2019']]

# Initialize the AddFIPS object
af = addfips.AddFIPS()

def add_fips_code(row):
    county_text = row['County']
    
    if '.' in county_text:
        parts = county_text.split('.')
        if len(parts) > 1:
            county_state = parts[1].strip()
        else:
            county_state = county_text
    else:
        county_state = county_text
    
    if ',' in county_state:
        county, state = county_state.split(',', 1)
        county = county.strip()
        state = state.strip()
    else:
        county = county_state
        state = None
    
    if "County" in county:
        county = county.replace(' County', '')
    
    if state and county:
        fips = af.get_county_fips(county, state=state)
        return fips
    else:
        return None

# Apply the function to create a new FIPS column
population['fips'] = population.apply(add_fips_code, axis=1)

# Reset index and melt the DataFrame
population_reset = population.reset_index(drop=True)
population_long = pd.melt(
    population_reset,
    id_vars=['County', 'fips'],
    value_vars=['2017', '2018', '2019'],
    var_name='year',
    value_name='population'
)

population_long['County'] = population_long['County'].str.replace(r'^\.\s*', '', regex=True)
population_long = population_long.sort_values(['County', 'year'])
population_long = population_long.reset_index(drop=True)
population_long['year'] = population_long['year'].astype(int)

# Clean employment data
columns_to_keep = ['area_fips','year','qtr','area_title','month1_emplvl','month2_emplvl','month3_emplvl']
employment_levels = employment_levels[columns_to_keep]

def create_county_fips(state_fips, municipal_code):
    state_fips_str = str(state_fips).zfill(2)
    municipal_code_str = str(municipal_code).zfill(3)
    county_fips = state_fips_str + municipal_code_str
    return county_fips

# Apply the Function to Create new FIPS column
zillow_county['CountyFIPS'] = zillow_county.apply(
    lambda row: create_county_fips(row['StateCodeFIPS'], row['MunicipalCodeFIPS']), 
    axis=1
)

# Filter for county data only
zillow_county = zillow_county[zillow_county['RegionType'] == 'county']

# Clean up Zillow county Data 
columns_to_drop = ['RegionID','SizeRank', 'RegionType','StateName','State','StateCodeFIPS','MunicipalCodeFIPS','Metro']
zillow_county = zillow_county.drop(columns = columns_to_drop)

# Melt the date columns into rows
melted_df = pd.melt(
    zillow_county,
    id_vars=['RegionName', 'CountyFIPS'],
    var_name='Date',
    value_name='HousingPrice'
)

melted_df['Date'] = pd.to_datetime(melted_df['Date'])
melted_df = melted_df.sort_values(['Date', 'RegionName'])
melted_df = melted_df.reset_index(drop=True)

# Filter to keep only data from 2017-2019
melted_df = melted_df[(melted_df['Date'] >= '2017-01-01') & (melted_df['Date'] <= '2019-12-31')]
melted_df = melted_df.reset_index(drop=True)

# Clean tariffs data frame
tariffs = tariffs[['time','area_fips','tariff']]
tariffs['time'] = pd.to_datetime(tariffs['time'])
tariffs['time'] = tariffs['time'] - timedelta(days=1)
filtered_tariffs = tariffs[(tariffs['time'] >= '2017-01-01') & (tariffs['time'] <= '2019-12-31')]
filtered_tariffs = filtered_tariffs.reset_index(drop=True)
filtered_tariffs['area_fips'] = filtered_tariffs['area_fips'].astype(str)

# Process employment data
county_employment_levels = employment_levels[employment_levels['area_title'].str.contains('County', case=False, na=False)]

transformed_data = []
for _, row in county_employment_levels.iterrows():
    area_fips = row['area_fips']
    area_title = row['area_title']
    year = row['year']
    quarter = row['qtr']
    
    base_month = (quarter - 1) * 3 + 1
    
    for month_idx in range(3):
        month_num = base_month + month_idx
        
        if month_num == 2:
            if (year % 4 == 0 and year % 100 != 0) or (year % 400 == 0):
                day = 29
            else:
                day = 28
        elif month_num in [4, 6, 9, 11]:
            day = 30
        else:
            day = 31
        
        date_str = f"{year}-{month_num:02d}-{day}"
        employment_level = row[f'month{month_idx+1}_emplvl']
        
        transformed_data.append({
            'CountyFIPS': area_fips,
            'RegionName': area_title,
            'Date': date_str,
            'EmploymentLevel': employment_level
        })

transformed_employment_df = pd.DataFrame(transformed_data)
transformed_employment_df['Date'] = pd.to_datetime(transformed_employment_df['Date'])
transformed_employment_df = transformed_employment_df.sort_values(['CountyFIPS', 'Date']).reset_index(drop=True)

# Merge all datasets
merged_df = pd.merge(
    transformed_employment_df,
    melted_df,
    how='inner',
    left_on=['CountyFIPS', 'Date'],
    right_on=['CountyFIPS', 'Date']
)

merged_df = pd.merge(
    merged_df,
    filtered_tariffs,
    how='inner',
    left_on=['CountyFIPS', 'Date'],
    right_on=['area_fips','time']
)

merged_df['year'] = merged_df['Date'].dt.year
merged_df = pd.merge(
    merged_df,
    population_long,
    how='inner',
    left_on=['CountyFIPS','year'],
    right_on=['fips','year']
)

# Clean merged data
merged_df = merged_df.drop(columns=['RegionName_x','RegionName_y','area_fips','year','time','County','fips'])
merged_df = merged_df.dropna()
merged_df = merged_df.reset_index(drop=True)

merged_df['population'] = merged_df['population'].astype(str).str.replace(',', '')
merged_df['population'] = pd.to_numeric(merged_df['population'], errors='coerce')
merged_df['HousingPrice'] = np.log(merged_df['HousingPrice'])

# NOW CREATE TARIFF QUARTILES LIKE IN HOUSING PRICES COUNTY PLOTS
print("Creating tariff quartiles...")

# Create tariff change data (similar to housing prices county plots)
tariffs_clean = filtered_tariffs.copy()
tariffs_clean['area_fips'] = tariffs_clean['area_fips'].astype(int)

# Pivot tariff data to calculate changes
tariffs_clean_pivoted = tariffs_clean.pivot(index='area_fips', columns='time', values='tariff')
tariffs_clean_pivoted = tariffs_clean_pivoted.reset_index()
tariffs_clean_pivoted = tariffs_clean_pivoted.dropna()

# Calculate tariff change (last - first)
tariffs_clean_pivoted['tariff_change'] = tariffs_clean_pivoted.iloc[:,-1] - tariffs_clean_pivoted.iloc[:,1]
tariffs_clean_pivoted = tariffs_clean_pivoted.sort_values(by=['tariff_change'], ascending=False)
tariffs_clean_pivoted = tariffs_clean_pivoted.reset_index(drop=True)

# Create quartiles based on tariff changes (Q1 = lowest change, Q4 = highest change)
tariffs_clean_pivoted['quartile'] = pd.qcut(tariffs_clean_pivoted['tariff_change'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])

print("Quartile distribution (Q1 = least affected, Q4 = most affected):")
print(tariffs_clean_pivoted['quartile'].value_counts().sort_index())
print("\nQuartile ranges:")
quartile_ranges = tariffs_clean_pivoted.groupby('quartile')['tariff_change'].agg(['min', 'max'])
print(quartile_ranges)

# Split into quartile dataframes
q1 = tariffs_clean_pivoted[tariffs_clean_pivoted['quartile'] == 'Q1']  # Least affected (lowest tariff increase)
q2 = tariffs_clean_pivoted[tariffs_clean_pivoted['quartile'] == 'Q2']
q3 = tariffs_clean_pivoted[tariffs_clean_pivoted['quartile'] == 'Q3']
q4 = tariffs_clean_pivoted[tariffs_clean_pivoted['quartile'] == 'Q4']  # Most affected (highest tariff increase)

# Convert area_fips to string for merging
merged_df['CountyFIPS_int'] = merged_df['CountyFIPS'].astype(int)

# REGRESSION ANALYSIS FUNCTIONS (from Housing Regression.py)
def create_12month_differences(df):
    """Calculate 12-month differences for all variables"""
    df = df.copy()
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values(['CountyFIPS', 'Date'])
    
    # Calculate 12-month lagged values by county
    df['HousingPrice_12m'] = df.groupby('CountyFIPS')['HousingPrice'].shift(12)
    df['EmploymentLevel_12m'] = df.groupby('CountyFIPS')['EmploymentLevel'].shift(12)
    df['tariff_12m'] = df.groupby('CountyFIPS')['tariff'].shift(12)
    df['population_12m'] = df.groupby('CountyFIPS')['population'].shift(12)
    
    # Calculate 12-month differences
    df['HousingPrice_diff'] = df['HousingPrice'] - df['HousingPrice_12m']
    
    # Employment: calculate log difference
    df = df[(df['EmploymentLevel'] > 0) & (df['EmploymentLevel_12m'] > 0)]
    df['EmploymentLevel_diff'] = np.log(df['EmploymentLevel']) - np.log(df['EmploymentLevel_12m'])
    
    # Tariff: calculate difference
    df['tariff_diff'] = df['tariff'] - df['tariff_12m']
    
    # Use the earlier year's population as weight
    df['weight_population'] = df['population_12m']
    
    # Drop rows with missing 12-month differences
    df = df.dropna(subset=['HousingPrice_diff', 'EmploymentLevel_diff', 'tariff_diff', 'weight_population'])
    
    # Keep only the columns we need
    result_df = df[['CountyFIPS', 'Date', 'HousingPrice_diff', 'tariff_diff', 
                   'EmploymentLevel_diff', 'weight_population']].copy()
    
    # Rename for consistency
    result_df = result_df.rename(columns={
        'HousingPrice_diff': 'HousingPrice',
        'tariff_diff': 'tariff',
        'EmploymentLevel_diff': 'EmploymentLevel',
        'weight_population': 'population'
    })
    
    return result_df

def create_dummy_variables(df):
    """Create county and date dummy variables for fixed effects"""
    # County dummies
    county_dummies = pd.get_dummies(df['CountyFIPS'], prefix='County')
    merged_data_county_dummies = pd.concat([df, county_dummies], axis=1)
    merged_data_county_dummies = merged_data_county_dummies.drop(columns=['Date', 'CountyFIPS', 'HousingPrice', 'population'])
    
    # Date dummies  
    date_dummies = pd.get_dummies(df['Date'], prefix='Date')
    merged_data_date_dummies = pd.concat([df, date_dummies], axis=1)
    merged_data_date_dummies = merged_data_date_dummies.drop(columns=['Date', 'CountyFIPS', 'HousingPrice', 'population'])
    
    # Combined dummies
    merged_data_large = pd.concat([merged_data_county_dummies, date_dummies], axis=1)
    
    return merged_data_county_dummies, merged_data_date_dummies, merged_data_large

def add_stars(coef, pval):
    """Add significance stars to coefficients"""
    if pval < 0.01:
        return f"{coef:.3f}***"
    elif pval < 0.05:
        return f"{coef:.3f}**"
    elif pval < 0.1:
        return f"{coef:.3f}*"
    else:
        return f"{coef:.3f}"

def run_regressions_for_quartile(data, quartile_name):
    """Run the 6 regressions for a specific quartile"""
    print(f"\nRunning regressions for {quartile_name}...")
    print(f"Number of observations: {len(data)}")
    print(f"Number of counties: {data['CountyFIPS'].nunique()}")
    
    # Create 12-month differences
    panel_data = create_12month_differences(data)
    
    if len(panel_data) == 0:
        print(f"No data available for {quartile_name} after creating differences")
        return None
    
    print(f"After creating differences - Observations: {len(panel_data)}, Counties: {panel_data['CountyFIPS'].nunique()}")
    
    # Create dummy variables
    merged_data_county_dummies, merged_data_date_dummies, merged_data_large = create_dummy_variables(panel_data)
    
    try:
        # Regression 1: Simple OLS with just tariff change
        reg_1 = sm.OLS(
            endog=panel_data['HousingPrice'],
            exog=sm.add_constant(panel_data['tariff'])
        ).fit()

        # Regression 2: WLS with population weights
        reg_2 = sm.WLS(
            endog=panel_data['HousingPrice'],
            exog=sm.add_constant(panel_data['tariff']),
            weights=panel_data['population']
        ).fit()

        # Regression 3: WLS with time dummies (but no employment change)
        reg_3 = sm.WLS(
            endog=panel_data['HousingPrice'],
            exog=sm.add_constant(merged_data_date_dummies.drop(columns=['EmploymentLevel'])),
            weights=panel_data['population']
        ).fit()

        # Regression 4: WLS with time dummies and employment change
        reg_4 = sm.WLS(
            endog=panel_data['HousingPrice'],
            exog=sm.add_constant(merged_data_date_dummies),
            weights=panel_data['population']
        ).fit()

        # Regression 5: WLS with county and time dummies (but no employment change)
        reg_5 = sm.WLS(
            endog=panel_data['HousingPrice'],
            exog=sm.add_constant(merged_data_large.drop(columns=['EmploymentLevel'])),
            weights=panel_data['population']
        ).fit()

        # Regression 6: WLS with county and time dummies and employment change
        reg_6 = sm.WLS(
            endog=panel_data['HousingPrice'],
            exog=sm.add_constant(merged_data_large),
            weights=panel_data['population']
        ).fit()

        return [reg_1, reg_2, reg_3, reg_4, reg_5, reg_6], panel_data
    
    except Exception as e:
        print(f"Error running regressions for {quartile_name}: {str(e)}")
        return None, panel_data

def create_regression_table(reg_list, quartile_name):
    """Create a formatted regression table"""
    if reg_list is None:
        return None
        
    # Create an empty dataframe for the table
    columns = ['(1)', '(2)', '(3)', '(4)', '(5)', '(6)']
    rows = ['Δ Tariff', 'Tariff SE', 'Δ Log Employment', 'Employment SE', 'Fixed Effects:', 'County', 'Time', 'Observations', 'R-squared']
    table = pd.DataFrame(index=rows, columns=columns)
    
    # Initialize all cells to empty string
    for col in columns:
        for row in rows:
            table.loc[row, col] = ""
    
    # Extract coefficient values with stars and standard errors
    # Tariff coefficients
    for i, reg in enumerate(reg_list):
        col = columns[i]
        if 'tariff' in reg.params:
            table.loc['Δ Tariff', col] = add_stars(reg.params['tariff'], reg.pvalues['tariff'])
            table.loc['Tariff SE', col] = f"({reg.bse['tariff']:.3f})"
    
    # Employment coefficients (for regressions that include it)
    employment_regs = [reg_list[3], reg_list[5]]  # reg_4 and reg_6
    employment_cols = [columns[3], columns[5]]
    
    for reg, col in zip(employment_regs, employment_cols):
        if 'EmploymentLevel' in reg.params:
            table.loc['Δ Log Employment', col] = add_stars(reg.params['EmploymentLevel'], reg.pvalues['EmploymentLevel'])
            table.loc['Employment SE', col] = f"({reg.bse['EmploymentLevel']:.3f})"
    
    # Fixed effects indicators
    table.loc['Fixed Effects:'] = ""
    table.loc['County'] = ['N', 'N', 'N', 'N', 'Y', 'Y']
    table.loc['Time'] = ['N', 'N', 'Y', 'Y', 'Y', 'Y']
    
    # Model statistics
    table.loc['Observations'] = [f"{int(reg.nobs)}" for reg in reg_list]
    table.loc['R-squared'] = [f"{reg.rsquared:.3f}" for reg in reg_list]
    
    return table

# RUN REGRESSIONS FOR EACH QUARTILE
quartile_results = {}
quartile_data = {
    'Q1 (Least Affected - Lowest Tariff Increase)': q1,
    'Q2 (Low-Medium Tariff Increase)': q2,
    'Q3 (Medium-High Tariff Increase)': q3,
    'Q4 (Most Affected - Highest Tariff Increase)': q4
}

for quartile_name, quartile_counties in quartile_data.items():
    print(f"\n{'='*60}")
    print(f"PROCESSING {quartile_name.upper()}")
    print(f"{'='*60}")
    
    # Filter merged_df to only include counties in this quartile
    quartile_merged_df = merged_df[merged_df['CountyFIPS_int'].isin(quartile_counties['area_fips'])]
    
    if len(quartile_merged_df) == 0:
        print(f"No matching data for {quartile_name}")
        continue
    
    # Run regressions for this quartile
    reg_results, panel_data = run_regressions_for_quartile(quartile_merged_df, quartile_name)
    
    if reg_results is not None:
        # Create and display table
        table = create_regression_table(reg_results, quartile_name)
        
        if table is not None:
            print(f"\nRegression Results for {quartile_name}:")
            print(table)
            
            # Store results
            quartile_results[quartile_name] = {
                'regressions': reg_results,
                'table': table,
                'panel_data': panel_data
            }
            
            # Print key statistics
            print(f"\nKey Statistics for {quartile_name}:")
            print(f"Average 12-month tariff change: {panel_data['tariff'].mean():.4f}")
            print(f"Average 12-month log price change: {panel_data['HousingPrice'].mean():.4f}")
            print(f"Tariff coefficient (full model): {reg_results[5].params['tariff']:.4f}")
            print(f"P-value (full model): {reg_results[5].pvalues['tariff']:.4f}")

# SUMMARY COMPARISON
print(f"\n{'='*80}")
print("SUMMARY COMPARISON ACROSS QUARTILES")
print(f"{'='*80}")

summary_data = []
for quartile_name, results in quartile_results.items():
    if results is not None:
        full_model = results['regressions'][5]  # The most complete model
        panel_data = results['panel_data']
        
        summary_data.append({
            'Quartile': quartile_name,
            'Counties': panel_data['CountyFIPS'].nunique(),
            'Observations': len(panel_data),
            'Avg Tariff Change': panel_data['tariff'].mean(),
            'Avg Price Change': panel_data['HousingPrice'].mean(),
            'Tariff Coefficient': full_model.params['tariff'],
            'P-value': full_model.pvalues['tariff'],
            'R-squared': full_model.rsquared
        })

summary_df = pd.DataFrame(summary_data)
print(summary_df.round(4))

# Save results
print(f"\n{'='*60}")
print("SAVING RESULTS")
print(f"{'='*60}")

# Create Images directory if it doesn't exist
os.makedirs('Images', exist_ok=True)

# Save individual tables
for quartile_name, results in quartile_results.items():
    if results is not None:
        filename = f"regression_table_{quartile_name.replace(' ', '_').replace('(', '').replace(')', '').lower()}.csv"
        results['table'].to_csv(f"Images/{filename}")
        print(f"Saved {filename}")

# Save summary table
summary_df.to_csv("Images/quartile_summary_comparison.csv", index=False)
print("Saved quartile_summary_comparison.csv")

print(f"\n{'='*60}")
print("ANALYSIS COMPLETE")
print(f"{'='*60}")

Creating tariff quartiles...
Quartile distribution (Q1 = least affected, Q4 = most affected):
Q1    808
Q2    808
Q3    808
Q4    808
Name: quartile, dtype: int64

Quartile ranges:
               min        max
quartile                     
Q1        0.000000   0.879202
Q2        0.879588   2.074396
Q3        2.074438   4.013297
Q4        4.018199  23.982700

PROCESSING Q1 (LEAST AFFECTED - LOWEST TARIFF INCREASE)

Running regressions for Q1 (Least Affected - Lowest Tariff Increase)...
Number of observations: 20836
Number of counties: 584
After creating differences - Observations: 13833, Counties: 579

Regression Results for Q1 (Least Affected - Lowest Tariff Increase):
                        (1)        (2)       (3)       (4)      (5)      (6)
Δ Tariff          -0.007***  -0.020***  0.039***  0.037***    0.002    0.001
Tariff SE           (0.002)    (0.002)   (0.003)   (0.003)  (0.003)  (0.003)
Δ Log Employment                                  0.234***            -0.011
Employment SE

In [19]:
#!/usr/bin/env python
# coding: utf-8

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from stargazer.stargazer import Stargazer
from datetime import timedelta, datetime
from IPython.display import display, Latex, Math, HTML
from linearmodels.panel import PanelOLS
import addfips
import warnings
import os

# Suppress warnings
warnings.filterwarnings('ignore')

#Read in data files
zillow_county = pd.read_csv('Data/zillow county.csv')
employment_levels = pd.read_csv('Data/employment_data_2017_2019_combined.csv')
tariffs = pd.read_csv('Data/ustariffs-by-county.csv')
population = pd.read_csv('Data/population_data.csv')

# Begin Editing population file
population.rename(columns={'Unnamed: 0': 'County'}, inplace=True)
population = population[['County','2017','2018','2019']]

# Initialize the AddFIPS object
af = addfips.AddFIPS()

def add_fips_code(row):
    county_text = row['County']
    
    if '.' in county_text:
        parts = county_text.split('.')
        if len(parts) > 1:
            county_state = parts[1].strip()
        else:
            county_state = county_text
    else:
        county_state = county_text
    
    if ',' in county_state:
        county, state = county_state.split(',', 1)
        county = county.strip()
        state = state.strip()
    else:
        county = county_state
        state = None
    
    if "County" in county:
        county = county.replace(' County', '')
    
    if state and county:
        fips = af.get_county_fips(county, state=state)
        return fips
    else:
        return None

# Apply the function to create a new FIPS column
population['fips'] = population.apply(add_fips_code, axis=1)

# Reset index and melt the DataFrame
population_reset = population.reset_index(drop=True)
population_long = pd.melt(
    population_reset,
    id_vars=['County', 'fips'],
    value_vars=['2017', '2018', '2019'],
    var_name='year',
    value_name='population'
)

population_long['County'] = population_long['County'].str.replace(r'^\.\s*', '', regex=True)
population_long = population_long.sort_values(['County', 'year'])
population_long = population_long.reset_index(drop=True)
population_long['year'] = population_long['year'].astype(int)

# Clean employment data
columns_to_keep = ['area_fips','year','qtr','area_title','month1_emplvl','month2_emplvl','month3_emplvl']
employment_levels = employment_levels[columns_to_keep]

def create_county_fips(state_fips, municipal_code):
    state_fips_str = str(state_fips).zfill(2)
    municipal_code_str = str(municipal_code).zfill(3)
    county_fips = state_fips_str + municipal_code_str
    return county_fips

# Apply the Function to Create new FIPS column
zillow_county['CountyFIPS'] = zillow_county.apply(
    lambda row: create_county_fips(row['StateCodeFIPS'], row['MunicipalCodeFIPS']), 
    axis=1
)

# Filter for county data only
zillow_county = zillow_county[zillow_county['RegionType'] == 'county']

# Clean up Zillow county Data 
columns_to_drop = ['RegionID','SizeRank', 'RegionType','StateName','State','StateCodeFIPS','MunicipalCodeFIPS','Metro']
zillow_county = zillow_county.drop(columns = columns_to_drop)

# Melt the date columns into rows
melted_df = pd.melt(
    zillow_county,
    id_vars=['RegionName', 'CountyFIPS'],
    var_name='Date',
    value_name='HousingPrice'
)

melted_df['Date'] = pd.to_datetime(melted_df['Date'])
melted_df = melted_df.sort_values(['Date', 'RegionName'])
melted_df = melted_df.reset_index(drop=True)

# Filter to keep only data from 2017-2019
melted_df = melted_df[(melted_df['Date'] >= '2017-01-01') & (melted_df['Date'] <= '2019-12-31')]
melted_df = melted_df.reset_index(drop=True)

# Clean tariffs data frame
tariffs = tariffs[['time','area_fips','tariff']]
tariffs['time'] = pd.to_datetime(tariffs['time'])
tariffs['time'] = tariffs['time'] - timedelta(days=1)
filtered_tariffs = tariffs[(tariffs['time'] >= '2017-01-01') & (tariffs['time'] <= '2019-12-31')]
filtered_tariffs = filtered_tariffs.reset_index(drop=True)
filtered_tariffs['area_fips'] = filtered_tariffs['area_fips'].astype(str)

# Process employment data
county_employment_levels = employment_levels[employment_levels['area_title'].str.contains('County', case=False, na=False)]

transformed_data = []
for _, row in county_employment_levels.iterrows():
    area_fips = row['area_fips']
    area_title = row['area_title']
    year = row['year']
    quarter = row['qtr']
    
    base_month = (quarter - 1) * 3 + 1
    
    for month_idx in range(3):
        month_num = base_month + month_idx
        
        if month_num == 2:
            if (year % 4 == 0 and year % 100 != 0) or (year % 400 == 0):
                day = 29
            else:
                day = 28
        elif month_num in [4, 6, 9, 11]:
            day = 30
        else:
            day = 31
        
        date_str = f"{year}-{month_num:02d}-{day}"
        employment_level = row[f'month{month_idx+1}_emplvl']
        
        transformed_data.append({
            'CountyFIPS': area_fips,
            'RegionName': area_title,
            'Date': date_str,
            'EmploymentLevel': employment_level
        })

transformed_employment_df = pd.DataFrame(transformed_data)
transformed_employment_df['Date'] = pd.to_datetime(transformed_employment_df['Date'])
transformed_employment_df = transformed_employment_df.sort_values(['CountyFIPS', 'Date']).reset_index(drop=True)

# Merge all datasets
merged_df = pd.merge(
    transformed_employment_df,
    melted_df,
    how='inner',
    left_on=['CountyFIPS', 'Date'],
    right_on=['CountyFIPS', 'Date']
)

merged_df = pd.merge(
    merged_df,
    filtered_tariffs,
    how='inner',
    left_on=['CountyFIPS', 'Date'],
    right_on=['area_fips','time']
)

merged_df['year'] = merged_df['Date'].dt.year
merged_df = pd.merge(
    merged_df,
    population_long,
    how='inner',
    left_on=['CountyFIPS','year'],
    right_on=['fips','year']
)

# Clean merged data
merged_df = merged_df.drop(columns=['RegionName_x','RegionName_y','area_fips','year','time','County','fips'])
merged_df = merged_df.dropna()
merged_df = merged_df.reset_index(drop=True)

merged_df['population'] = merged_df['population'].astype(str).str.replace(',', '')
merged_df['population'] = pd.to_numeric(merged_df['population'], errors='coerce')
merged_df['HousingPrice'] = np.log(merged_df['HousingPrice'])

# NOW CREATE TARIFF QUARTILES LIKE IN HOUSING PRICES COUNTY PLOTS
print("Creating tariff quartiles...")

# Create tariff change data (similar to housing prices county plots)
tariffs_clean = filtered_tariffs.copy()
tariffs_clean['area_fips'] = tariffs_clean['area_fips'].astype(int)

# Pivot tariff data to calculate changes
tariffs_clean_pivoted = tariffs_clean.pivot(index='area_fips', columns='time', values='tariff')
tariffs_clean_pivoted = tariffs_clean_pivoted.reset_index()
tariffs_clean_pivoted = tariffs_clean_pivoted.dropna()

# Calculate tariff change (last - first)
tariffs_clean_pivoted['tariff_change'] = tariffs_clean_pivoted.iloc[:,-1] - tariffs_clean_pivoted.iloc[:,1]
tariffs_clean_pivoted = tariffs_clean_pivoted.sort_values(by=['tariff_change'], ascending=False)
tariffs_clean_pivoted = tariffs_clean_pivoted.reset_index(drop=True)

# Create quartiles based on tariff changes (Q1 = lowest change, Q4 = highest change)
tariffs_clean_pivoted['quartile'] = pd.qcut(tariffs_clean_pivoted['tariff_change'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])

print("Quartile distribution (Q1 = least affected, Q4 = most affected):")
print(tariffs_clean_pivoted['quartile'].value_counts().sort_index())
print("\nQuartile ranges:")
quartile_ranges = tariffs_clean_pivoted.groupby('quartile')['tariff_change'].agg(['min', 'max'])
print(quartile_ranges)

# Split into quartile dataframes
q1 = tariffs_clean_pivoted[tariffs_clean_pivoted['quartile'] == 'Q1']  # Least affected (lowest tariff increase)
q2 = tariffs_clean_pivoted[tariffs_clean_pivoted['quartile'] == 'Q2']
q3 = tariffs_clean_pivoted[tariffs_clean_pivoted['quartile'] == 'Q3']
q4 = tariffs_clean_pivoted[tariffs_clean_pivoted['quartile'] == 'Q4']  # Most affected (highest tariff increase)

# Convert area_fips to string for merging
merged_df['CountyFIPS_int'] = merged_df['CountyFIPS'].astype(int)

# REGRESSION ANALYSIS FUNCTIONS (from Housing Regression.py)
def create_12month_differences(df):
    """Calculate 12-month differences for all variables"""
    df = df.copy()
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values(['CountyFIPS', 'Date'])
    
    # Calculate 12-month lagged values by county
    df['HousingPrice_12m'] = df.groupby('CountyFIPS')['HousingPrice'].shift(12)
    df['EmploymentLevel_12m'] = df.groupby('CountyFIPS')['EmploymentLevel'].shift(12)
    df['tariff_12m'] = df.groupby('CountyFIPS')['tariff'].shift(12)
    df['population_12m'] = df.groupby('CountyFIPS')['population'].shift(12)
    
    # Calculate 12-month differences
    df['HousingPrice_diff'] = df['HousingPrice'] - df['HousingPrice_12m']
    
    # Employment: calculate log difference
    df = df[(df['EmploymentLevel'] > 0) & (df['EmploymentLevel_12m'] > 0)]
    df['EmploymentLevel_diff'] = np.log(df['EmploymentLevel']) - np.log(df['EmploymentLevel_12m'])
    
    # Tariff: calculate difference
    df['tariff_diff'] = df['tariff'] - df['tariff_12m']
    
    # Use the earlier year's population as weight
    df['weight_population'] = df['population_12m']
    
    # Drop rows with missing 12-month differences
    df = df.dropna(subset=['HousingPrice_diff', 'EmploymentLevel_diff', 'tariff_diff', 'weight_population'])
    
    # Keep only the columns we need
    result_df = df[['CountyFIPS', 'Date', 'HousingPrice_diff', 'tariff_diff', 
                   'EmploymentLevel_diff', 'weight_population']].copy()
    
    # Rename for consistency
    result_df = result_df.rename(columns={
        'HousingPrice_diff': 'HousingPrice',
        'tariff_diff': 'tariff',
        'EmploymentLevel_diff': 'EmploymentLevel',
        'weight_population': 'population'
    })
    
    return result_df

def create_dummy_variables(df):
    """Create county and date dummy variables for fixed effects"""
    # County dummies
    county_dummies = pd.get_dummies(df['CountyFIPS'], prefix='County')
    merged_data_county_dummies = pd.concat([df, county_dummies], axis=1)
    merged_data_county_dummies = merged_data_county_dummies.drop(columns=['Date', 'CountyFIPS', 'HousingPrice', 'population'])
    
    # Date dummies  
    date_dummies = pd.get_dummies(df['Date'], prefix='Date')
    merged_data_date_dummies = pd.concat([df, date_dummies], axis=1)
    merged_data_date_dummies = merged_data_date_dummies.drop(columns=['Date', 'CountyFIPS', 'HousingPrice', 'population'])
    
    # Combined dummies
    merged_data_large = pd.concat([merged_data_county_dummies, date_dummies], axis=1)
    
    return merged_data_county_dummies, merged_data_date_dummies, merged_data_large

def add_stars(coef, pval):
    """Add significance stars to coefficients"""
    if pval < 0.01:
        return f"{coef:.3f}***"
    elif pval < 0.05:
        return f"{coef:.3f}**"
    elif pval < 0.1:
        return f"{coef:.3f}*"
    else:
        return f"{coef:.3f}"

def create_latex_table(reg_list, quartile_name, quartile_label):
    """Create a LaTeX formatted regression table"""
    if reg_list is None:
        return None
    
    latex_code = f"""\\begin{{table}}[htbp]
\\centering
\\caption{{Effect of Tariff Changes on 12-Month Housing Price Changes - {quartile_name}}}
\\label{{tab:tariffs_changes_{quartile_label}}}
\\begin{{tabular}}{{lcccccc}}
\\toprule
{{}} &       (1) &      (2) &       (3) &       (4) &       (5) &        (6) \\\\
\\midrule"""
    
    # Tariff coefficients
    tariff_row = "$\\Delta$ Tariff  &  "
    se_row = "                 &  "
    
    for i, reg in enumerate(reg_list):
        if 'tariff' in reg.params:
            coef_with_stars = add_stars(reg.params['tariff'], reg.pvalues['tariff'])
            se_val = f"({reg.bse['tariff']:.3f})"
        else:
            coef_with_stars = ""
            se_val = ""
        
        if i < len(reg_list) - 1:
            tariff_row += f"{coef_with_stars:>9} & "
            se_row += f"{se_val:>9} & "
        else:
            tariff_row += f"{coef_with_stars:>10} \\\\"
            se_row += f"{se_val:>10} \\\\"
    
    latex_code += f"\n{tariff_row}\n{se_row}"
    
    # Employment coefficients (only for regressions 4 and 6)
    emp_row = "$\\Delta$ Log Employment & "
    emp_se_row = "                 & "
    
    for i in range(6):
        if i == 3 or i == 5:  # regressions 4 and 6 (0-indexed)
            reg = reg_list[i]
            if 'EmploymentLevel' in reg.params:
                coef_with_stars = add_stars(reg.params['EmploymentLevel'], reg.pvalues['EmploymentLevel'])
                se_val = f"({reg.bse['EmploymentLevel']:.3f})"
            else:
                coef_with_stars = ""
                se_val = ""
        else:
            coef_with_stars = ""
            se_val = ""
        
        if i < 5:
            emp_row += f"{coef_with_stars:>9} & "
            emp_se_row += f"{se_val:>9} & "
        else:
            emp_row += f"{coef_with_stars:>10} \\\\"
            emp_se_row += f"{se_val:>10} \\\\"
    
    latex_code += f"\n{emp_row}\n{emp_se_row}"
    
    # Fixed effects
    latex_code += """
Fixed Effects:   &           &          &           &           &           &            \\\\
County           &         N &        N &         N &         N &         Y &          Y \\\\
Time             &         N &        N &         Y &         Y &         Y &          Y \\\\
Population Weight &        N &        Y &         Y &         Y &         Y &          Y \\\\
\\midrule"""
    
    # Model statistics
    obs_row = "Observations     & "
    rsq_row = "R-squared        & "
    
    for i, reg in enumerate(reg_list):
        obs_val = f"{int(reg.nobs)}"
        rsq_val = f"{reg.rsquared:.3f}"
        
        if i < len(reg_list) - 1:
            obs_row += f"{obs_val:>9} & "
            rsq_row += f"{rsq_val:>9} & "
        else:
            obs_row += f"{obs_val:>10} \\\\"
            rsq_row += f"{rsq_val:>10} \\\\"
    
    latex_code += f"\n{obs_row}\n{rsq_row}"
    
    latex_code += """
\\bottomrule
\\end{tabular}
\\end{table}"""
    
    return latex_code

def run_regressions_for_quartile(data, quartile_name):
    """Run the 6 regressions for a specific quartile"""
    print(f"\nRunning regressions for {quartile_name}...")
    print(f"Number of observations: {len(data)}")
    print(f"Number of counties: {data['CountyFIPS'].nunique()}")
    
    # Create 12-month differences
    panel_data = create_12month_differences(data)
    
    if len(panel_data) == 0:
        print(f"No data available for {quartile_name} after creating differences")
        return None, None
    
    print(f"After creating differences - Observations: {len(panel_data)}, Counties: {panel_data['CountyFIPS'].nunique()}")
    
    # Create dummy variables
    merged_data_county_dummies, merged_data_date_dummies, merged_data_large = create_dummy_variables(panel_data)
    
    try:
        # Regression 1: Simple OLS with just tariff change
        reg_1 = sm.OLS(
            endog=panel_data['HousingPrice'],
            exog=sm.add_constant(panel_data['tariff'])
        ).fit()

        # Regression 2: WLS with population weights
        reg_2 = sm.WLS(
            endog=panel_data['HousingPrice'],
            exog=sm.add_constant(panel_data['tariff']),
            weights=panel_data['population']
        ).fit()

        # Regression 3: WLS with time dummies (but no employment change)
        reg_3 = sm.WLS(
            endog=panel_data['HousingPrice'],
            exog=sm.add_constant(merged_data_date_dummies.drop(columns=['EmploymentLevel'])),
            weights=panel_data['population']
        ).fit()

        # Regression 4: WLS with time dummies and employment change
        reg_4 = sm.WLS(
            endog=panel_data['HousingPrice'],
            exog=sm.add_constant(merged_data_date_dummies),
            weights=panel_data['population']
        ).fit()

        # Regression 5: WLS with county and time dummies (but no employment change)
        reg_5 = sm.WLS(
            endog=panel_data['HousingPrice'],
            exog=sm.add_constant(merged_data_large.drop(columns=['EmploymentLevel'])),
            weights=panel_data['population']
        ).fit()

        # Regression 6: WLS with county and time dummies and employment change
        reg_6 = sm.WLS(
            endog=panel_data['HousingPrice'],
            exog=sm.add_constant(merged_data_large),
            weights=panel_data['population']
        ).fit()

        return [reg_1, reg_2, reg_3, reg_4, reg_5, reg_6], panel_data
    
    except Exception as e:
        print(f"Error running regressions for {quartile_name}: {str(e)}")
        return None, panel_data

def create_summary_latex_table(quartile_results):
    """Create a LaTeX summary table comparing all quartiles"""
    latex_code = """\\begin{table}[htbp]
\\centering
\\caption{Summary of Tariff Effects Across Quartiles (Full Model Specification)}
\\label{tab:quartile_summary}
\\begin{tabular}{lcccc}
\\toprule
{} & Q1 (Least) & Q2 (Low-Med) & Q3 (Med-High) & Q4 (Most) \\\\
\\midrule"""
    
    # Extract data for each quartile
    quartile_names = ['Q1 (Least Affected - Lowest Tariff Increase)', 
                     'Q2 (Low-Medium Tariff Increase)', 
                     'Q3 (Medium-High Tariff Increase)', 
                     'Q4 (Most Affected - Highest Tariff Increase)']
    
    # Tariff coefficients row
    coef_row = "$\\Delta$ Tariff Coefficient & "
    se_row = "                            & "
    pval_row = "P-value                     & "
    avg_tariff_row = "Average Tariff Change       & "
    avg_price_row = "Average Price Change        & "
    counties_row = "Counties                    & "
    obs_row = "Observations                & "
    rsq_row = "R-squared                   & "
    
    for i, qname in enumerate(quartile_names):
        if qname in quartile_results and quartile_results[qname] is not None:
            full_model = quartile_results[qname]['regressions'][5]  # The most complete model
            panel_data = quartile_results[qname]['panel_data']
            
            coef_with_stars = add_stars(full_model.params['tariff'], full_model.pvalues['tariff'])
            se_val = f"({full_model.bse['tariff']:.3f})"
            pval_val = f"{full_model.pvalues['tariff']:.3f}"
            avg_tariff_val = f"{panel_data['tariff'].mean():.3f}"
            avg_price_val = f"{panel_data['HousingPrice'].mean():.3f}"
            counties_val = f"{panel_data['CountyFIPS'].nunique()}"
            obs_val = f"{int(full_model.nobs)}"
            rsq_val = f"{full_model.rsquared:.3f}"
        else:
            coef_with_stars = "N/A"
            se_val = ""
            pval_val = "N/A"
            avg_tariff_val = "N/A"
            avg_price_val = "N/A"
            counties_val = "N/A"
            obs_val = "N/A"
            rsq_val = "N/A"
        
        if i < 3:
            coef_row += f"{coef_with_stars:>11} & "
            se_row += f"{se_val:>11} & "
            pval_row += f"{pval_val:>11} & "
            avg_tariff_row += f"{avg_tariff_val:>11} & "
            avg_price_row += f"{avg_price_val:>11} & "
            counties_row += f"{counties_val:>11} & "
            obs_row += f"{obs_val:>11} & "
            rsq_row += f"{rsq_val:>11} & "
        else:
            coef_row += f"{coef_with_stars:>9} \\\\"
            se_row += f"{se_val:>9} \\\\"
            pval_row += f"{pval_val:>9} \\\\"
            avg_tariff_row += f"{avg_tariff_val:>9} \\\\"
            avg_price_row += f"{avg_price_val:>9} \\\\"
            counties_row += f"{counties_val:>9} \\\\"
            obs_row += f"{obs_val:>9} \\\\"
            rsq_row += f"{rsq_val:>9} \\\\"
    
    latex_code += f"\n{coef_row}\n{se_row}\n{pval_row}\n{avg_tariff_row}\n{avg_price_row}\n{counties_row}\n{obs_row}\n{rsq_row}"
    
    latex_code += """
\\bottomrule
\\end{tabular}
\\end{table}"""
    
    return latex_code

# RUN REGRESSIONS FOR EACH QUARTILE
quartile_results = {}
quartile_data = {
    'Q1 (Least Affected - Lowest Tariff Increase)': q1,
    'Q2 (Low-Medium Tariff Increase)': q2,
    'Q3 (Medium-High Tariff Increase)': q3,
    'Q4 (Most Affected - Highest Tariff Increase)': q4
}

# Create LaTeX output directory
os.makedirs('LaTeX_Tables', exist_ok=True)

for quartile_name, quartile_counties in quartile_data.items():
    print(f"\n{'='*60}")
    print(f"PROCESSING {quartile_name.upper()}")
    print(f"{'='*60}")
    
    # Filter merged_df to only include counties in this quartile
    quartile_merged_df = merged_df[merged_df['CountyFIPS_int'].isin(quartile_counties['area_fips'])]
    
    if len(quartile_merged_df) == 0:
        print(f"No matching data for {quartile_name}")
        continue
    
    # Run regressions for this quartile
    reg_results, panel_data = run_regressions_for_quartile(quartile_merged_df, quartile_name)
    
    if reg_results is not None:
        # Create LaTeX table
        quartile_label = quartile_name.split()[0].lower()  # e.g., 'q1'
        latex_table = create_latex_table(reg_results, quartile_name, quartile_label)
        
        if latex_table is not None:
            # Store results
            quartile_results[quartile_name] = {
                'regressions': reg_results,
                'latex_table': latex_table,
                'panel_data': panel_data
            }
            
            # Save LaTeX table to file
            filename = f"LaTeX_Tables/regression_table_{quartile_label}.tex"
            with open(filename, 'w') as f:
                f.write(latex_table)
            print(f"Saved LaTeX table: {filename}")
            
            # Print LaTeX table to console
            print(f"\nLaTeX Table for {quartile_name}:")
            print(latex_table)
            
            # Print key statistics
            print(f"\nKey Statistics for {quartile_name}:")
            print(f"Average 12-month tariff change: {panel_data['tariff'].mean():.4f}")
            print(f"Average 12-month log price change: {panel_data['HousingPrice'].mean():.4f}")
            print(f"Tariff coefficient (full model): {reg_results[5].params['tariff']:.4f}")
            print(f"P-value (full model): {reg_results[5].pvalues['tariff']:.4f}")

# CREATE AND SAVE SUMMARY TABLE
summary_latex = create_summary_latex_table(quartile_results)
with open('LaTeX_Tables/summary_table.tex', 'w') as f:
    f.write(summary_latex)

print(f"\n{'='*80}")
print("SUMMARY LATEX TABLE")
print(f"{'='*80}")
print(summary_latex)

# SUMMARY COMPARISON (existing code)
print(f"\n{'='*80}")
print("SUMMARY COMPARISON ACROSS QUARTILES")
print(f"{'='*80}")

summary_data = []
for quartile_name, results in quartile_results.items():
    if results is not None:
        full_model = results['regressions'][5]  # The most complete model
        panel_data = results['panel_data']
        
        summary_data.append({
            'Quartile': quartile_name,
            'Counties': panel_data['CountyFIPS'].nunique(),
            'Observations': len(panel_data),
            'Avg Tariff Change': panel_data['tariff'].mean(),
            'Avg Price Change': panel_data['HousingPrice'].mean(),
            'Tariff Coefficient': full_model.params['tariff'],
            'P-value': full_model.pvalues['tariff'],
            'R-squared': full_model.rsquared
        })

summary_df = pd.DataFrame(summary_data)
print(summary_df.round(4))

# Save results
print(f"\n{'='*60}")
print("SAVING RESULTS")
print(f"{'='*60}")

# Create Images directory if it doesn't exist (for backward compatibility)
os.makedirs('Images', exist_ok=True)

# Save individual tables (CSV format for backward compatibility)
for quartile_name, results in quartile_results.items():
    if results is not None:
        # Create a simple table for CSV output
        reg_list = results['regressions']
        
        # Create an empty dataframe for the table
        columns = ['(1)', '(2)', '(3)', '(4)', '(5)', '(6)']
        rows = ['Δ Tariff', 'Tariff SE', 'Δ Log Employment', 'Employment SE', 'Fixed Effects:', 'County', 'Time', 'Observations', 'R-squared']
        table = pd.DataFrame(index=rows, columns=columns)
        
        # Initialize all cells to empty string
        for col in columns:
            for row in rows:
                table.loc[row, col] = ""
        
        # Extract coefficient values with stars and standard errors
        # Tariff coefficients
        for i, reg in enumerate(reg_list):
            col = columns[i]
            if 'tariff' in reg.params:
                table.loc['Δ Tariff', col] = add_stars(reg.params['tariff'], reg.pvalues['tariff'])
                table.loc['Tariff SE', col] = f"({reg.bse['tariff']:.3f})"
        
        # Employment coefficients (for regressions that include it)
        employment_regs = [reg_list[3], reg_list[5]]  # reg_4 and reg_6
        employment_cols = [columns[3], columns[5]]
        
        for reg, col in zip(employment_regs, employment_cols):
            if 'EmploymentLevel' in reg.params:
                table.loc['Δ Log Employment', col] = add_stars(reg.params['EmploymentLevel'], reg.pvalues['EmploymentLevel'])
                table.loc['Employment SE', col] = f"({reg.bse['EmploymentLevel']:.3f})"
        
        # Fixed effects indicators
        table.loc['Fixed Effects:'] = ""
        table.loc['County'] = ['N', 'N', 'N', 'N', 'Y', 'Y']
        table.loc['Time'] = ['N', 'N', 'Y', 'Y', 'Y', 'Y']
        
        # Model statistics
        table.loc['Observations'] = [f"{int(reg.nobs)}" for reg in reg_list]
        table.loc['R-squared'] = [f"{reg.rsquared:.3f}" for reg in reg_list]
        
        filename = f"regression_table_{quartile_name.replace(' ', '_').replace('(', '').replace(')', '').lower()}.csv"
        table.to_csv(f"Images/{filename}")
        print(f"Saved {filename}")

# Save summary table
summary_df.to_csv("Images/quartile_summary_comparison.csv", index=False)
print("Saved quartile_summary_comparison.csv")

print(f"\n{'='*60}")
print("LATEX TABLES SAVED")
print(f"{'='*60}")
print("All LaTeX tables have been saved to the 'LaTeX_Tables' directory:")
print("- regression_table_q1.tex")
print("- regression_table_q2.tex") 
print("- regression_table_q3.tex")
print("- regression_table_q4.tex")
print("- summary_table.tex")

print(f"\n{'='*60}")
print("ANALYSIS COMPLETE")
print(f"{'='*60}")

print("\nTo use the LaTeX tables in your document, make sure to include:")
print("\\usepackage{booktabs}")
print("\\usepackage{array}")
print("\nThen simply \\input{LaTeX_Tables/regression_table_q1.tex} etc. in your document.")

Creating tariff quartiles...
Quartile distribution (Q1 = least affected, Q4 = most affected):
Q1    808
Q2    808
Q3    808
Q4    808
Name: quartile, dtype: int64

Quartile ranges:
               min        max
quartile                     
Q1        0.000000   0.879202
Q2        0.879588   2.074396
Q3        2.074438   4.013297
Q4        4.018199  23.982700

PROCESSING Q1 (LEAST AFFECTED - LOWEST TARIFF INCREASE)

Running regressions for Q1 (Least Affected - Lowest Tariff Increase)...
Number of observations: 20836
Number of counties: 584
After creating differences - Observations: 13833, Counties: 579
Saved LaTeX table: LaTeX_Tables/regression_table_q1.tex

LaTeX Table for Q1 (Least Affected - Lowest Tariff Increase):
\begin{table}[htbp]
\centering
\caption{Effect of Tariff Changes on 12-Month Housing Price Changes - Q1 (Least Affected - Lowest Tariff Increase)}
\label{tab:tariffs_changes_q1}
\begin{tabular}{lcccccc}
\toprule
{} &       (1) &      (2) &       (3) &       (4) &       (5